# Notebook 11 — large-v3: zero-shot di dev + probe training di T4

> [!info] Satu run, dua pertanyaan, **tanpa training dan tanpa push ke HF**
> Langkah 1 dari rencana 30 Sep ([[next-steps-2026-09-30]]): go/no-go sebelum menghabiskan
> 6–8 jam GPU untuk stage 1 large-v3.

**Pertanyaan 1 — apakah `whisper-large-v3` penuh mendengar lebih baik dari turbo?**
Semua model kita berawal dari turbo (decoder 4 layer). large-v3 memakai encoder yang sama
ukurannya tapi decoder 32 layer, dan belum pernah diukur di proyek ini (NB 03 hanya tiny–medium
dan turbo). Evaluasi gap #1 menaruh ~0,14 dari WER dev D di bucket *pendengaran/leksikal*:
kalau ada tuas yang bisa menutup jarak besar, tuas itu harus menyerang bucket ini.

Tiga arm, semuanya lewat `main.py` D **byte demi byte** (beam 5), di 372 klip dev:

| arm | model | peran |
|---|---|---|
| D | E4-native (submission D) | **kontrol paritas**: harus tepat 3.965 error seperti NB 08 |
| T0 | `openai/whisper-large-v3-turbo`, zero-shot | titik awal semua fine-tune kita |
| L0 | `openai/whisper-large-v3`, zero-shot | kandidat |

Zero-shot = kedua model belum pernah melihat dev, jadi seluruh 372 klip sah sebagai data uji.
Pembanding utama adalah **ruang huruf kecil tanpa tanda baca**: model zero-shot belum mengenal
konvensi kapital/tanda baca referensi, dan itu yang nanti diurus tahap 2. Yang diukur di sini
adalah pendengaran.

**Pertanyaan 2 — muatkah large-v3 dilatih penuh di T4 16 GB, dan berapa lama?**
Probe sintetis (audio acak 30 s, label 128 token): puncak VRAM dan detik per micro-batch untuk
full fine-tune dan untuk varian encoder dibekukan, dengan resep stage 1 (adafactor, fp16,
gradient checkpointing, SpecAugment). Turbo ikut diprobe sebagai **kalibrasi**, karena angka
nyatanya diketahui dari NB 06 E4-native: 2,27 s per micro-batch 4, puncak 7,03 GB.

## Aturan keputusan (ditetapkan sebelum run)

**Pendengaran** — L0 − T0 berpasangan, ruang lower+tanpa tanda baca, 372 klip:
- CI95 seluruhnya < 0 → **L0 LEBIH BAIK** → lanjut ke stage 1 large-v3 (kalau kelayakan lolos).
- CI95 memuat 0 → **SERI** → large-v3 jadi taruhan tanpa dukungan; diputuskan bersama user.
- CI95 seluruhnya > 0 → **L0 LEBIH BURUK** → large-v3 dihentikan.

**Kelayakan** (dibaca bersama, bukan pass/fail notebook):
- Proyeksi waktu decode test di A100 ≤ 3.600 s (setengah batas 2 jam), diskalakan dari D.
- Load model ≤ 20 s (smoke test 1 menit juga memuat import).
- Ada konfigurasi full fine-tune dengan puncak ≤ 14,5 GB, dan proyeksi training stage 1
  (resep E4-native, 421 step) ≤ 9 jam = muat satu sesi Kaggle 12 jam bersama evaluasi.
  Kalau hanya varian encoder-beku yang muat, itu variabel kedua, jadi dibahas dulu.

**Perkiraan waktu: ~50–70 menit di T4.** Unduh ~8 GB, decode D ~6 menit, T0 ~6–10 menit,
L0 ~20–30 menit, probe ~10 menit. Settings: **GPU T4** (x2 boleh, notebook memakai satu),
Internet ON, secret `HF_TOKEN` (D ada di repo private), dataset `indonesian_dev`.

## 1 — Knob, lalu pin `transformers<5`

In [ ]:
# ============================================================
# CELL 1 - KNOBS, lalu pin transformers<5 SEBELUM impor apa pun
# ============================================================
import os, sys, subprocess
# Satu GPU: main.py dan probe sama-sama memakai cuda:0 (alasan: NB 06 sel 1).
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

NOTEBOOK_ID = "11-large-v3-probe"
RUN_NOTE    = "zeroshot-feasibility"
RUN_ID      = NOTEBOOK_ID + "-" + RUN_NOTE

# rev=None: resolusi revisi main saat ini dicatat di digest (model_info().sha), bersama sha16 bobotnya.
ARMS_SPEC = {
    "D":  dict(repo="amozaurus/mozilla-jv-06-data-mixing-E4-native",
               rev="3437f4424cdee0ce3113f7fe2ccbc32e5350be7e", sha16="b3438f32cf61ebbe",
               what="kontrol: E4-native = submission D, harus = NB 08"),
    "T0": dict(repo="openai/whisper-large-v3-turbo", rev=None, sha16=None,
               what="turbo zero-shot (titik awal semua fine-tune kita)"),
    "L0": dict(repo="openai/whisper-large-v3", rev=None, sha16=None,
               what="large-v3 zero-shot (kandidat)"),
}
RUN_ARMS  = ("D", "T0", "L0")
NUM_BEAMS = 5                 # decode = decode D yang dikirim

# ---- probe training (sintetis, tanpa data, tanpa push) ----
RUN_PROBE = True
# (arm, varian, micro-batch). Untuk "full", micro-batch diturunkan hanya kalau yang lebih
# besar OOM. T0 dulu: itulah kalibrasinya.
PROBE_CONFIGS = [("T0", "full", 4),
                 ("L0", "full", 4), ("L0", "full", 2), ("L0", "full", 1),
                 ("L0", "freeze_encoder", 4)]
PROBE_LABEL_LEN = 128         # token label per contoh (klip 20-25 s bicara cepat)
PROBE_MICRO     = 6           # micro-batch yang diukur, sesudah 1 pemanasan + 1 step optimizer
LEARNING_RATE   = 6e-6        # stage 1 E4-native
EFF_BATCH       = 16          # batch efektif stage 1
STAGE1_STEPS    = 421         # E4-native: 6.725 item, batch efektif 16, 1 epoch
VRAM_LIMIT_GB   = 14.5        # T4 15,1 GB terpakai; sisakan ruang fragmentasi
SEED            = 20260930

# ---- angka pembanding (dari findings/ dan submissions/, agregat saja) ----
D_REF = {"all": (3965, 17485), "2": (799, 3556), "5": (3166, 13929)}
REF_PATHS = dict(short=328, chunked=44, windows=88)
BASE_MAIN_SHA16 = "8f76b71382f9b33c"   # main.py dengan NUM_BEAMS = 1 (NB 08)
D_MAIN_SHA16    = "aa5a691f68011b81"   # main.py D yang dikirim (submission 2693)
PARITY_TOL = 0.005
N_BOOT     = 5000
KNOWN = dict(
    turbo_sec_per_micro_b4=2.2703,     # NB 06 E4-native, T4, micro-batch 4, data nyata
    turbo_peak_gb_b4=7.03,
    turbo_stage1_train_only_h=1.17,    # proyeksi smoke NB 06 (tanpa evaluasi)
    turbo_stage1_wall_h=2.72,          # run nyata, termasuk evaluasi dalam training
    D_test_a100_s=619.9,               # submission 2693: 2.118 klip di A100
    D_smoke_total_s=17.9,
    D_test=0.2901, F_test=0.2660, leaderboard_1=0.2076)
TEST_TARGET_S = 3600
LOAD_TARGET_S = 20.0

assert RUN_NOTE, "RUN_NOTE wajib diisi (CLAUDE.md aturan 3)"
assert RUN_ARMS[0] == "D", "D harus di-decode pertama: waktu arm lain diskalakan darinya"

r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers<5"],
                   capture_output=True, text=True)
print("pip transformers<5 ->", "OK" if r.returncode == 0 else "GAGAL")
if r.returncode:
    print(r.stderr[-1500:])

import platform
import numpy, torch, transformers
print("python       :", platform.python_version())
print("transformers :", transformers.__version__, " (container: <5)")
print("torch        :", torch.__version__)
print("numpy        :", numpy.__version__)
print("gpu          :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "TIDAK ADA")
MAJOR = int(transformers.__version__.split(".")[0])
if MAJOR >= 5:
    raise RuntimeError("transformers %s masih >= 5: pin gagal, atau transformers sudah diimpor "
                       "sebelum sel ini. Run -> Restart & clear, lalu Run All." % transformers.__version__)
if not torch.cuda.is_available():
    raise RuntimeError("GPU mati. Settings -> Accelerator -> GPU T4, lalu Run All lagi.")
GPU_TOTAL_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
print("vram         : %.1f GB" % GPU_TOTAL_GB)
print("\nRUN_ID =", RUN_ID, "| arm", RUN_ARMS, "| beam", NUM_BEAMS, "| probe", RUN_PROBE)

## 2 — Output, log, discovery, HuggingFace

In [ ]:
# ============================================================
# CELL 2 - OUTPUT, LOG, DISCOVERY, HUGGINGFACE
# ============================================================
import json, re, csv, time, math, random, shutil, hashlib, collections, gc, statistics
from pathlib import Path
import numpy as np, pandas as pd
import warnings
warnings.filterwarnings("ignore")
random.seed(SEED); np.random.seed(SEED)

def head(t):
    print("\n" + "=" * 74); print(t); print("=" * 74)

# Cermin /kaggle/working/findings/<RUN_ID>/ <-> <repo>/findings/<RUN_ID>/. Bobot tidak pernah
# masuk /kaggle/working (itu yang dipaketkan "Download all"); semuanya di /kaggle/temp.
WORKING = Path("/kaggle/working")
TEMP    = Path("/kaggle/temp"); TEMP.mkdir(parents=True, exist_ok=True)
OUT_DIR = WORKING / "findings" / RUN_ID
OUT_DIR.mkdir(parents=True, exist_ok=True)

def out(name):
    return OUT_DIR / name

_ORIG_STDOUT = getattr(sys, "_nb_orig_stdout", sys.stdout)
sys._nb_orig_stdout = _ORIG_STDOUT

class _Tee:
    def __init__(self, path, stream):
        self.f = open(path, "w", encoding="utf-8"); self.s = stream
    def write(self, t):
        self.s.write(t); self.f.write(t); return len(t)
    def flush(self):
        try: self.s.flush()
        except Exception: pass
        self.f.flush()
    def isatty(self): return False

try:
    if isinstance(sys.stdout, _Tee): sys.stdout.f.close()
except Exception:
    pass
sys.stdout = _Tee(out("run-log.txt"), _ORIG_STDOUT)

# Digest untuk Claude: AGREGAT SAJA. Tidak ada nama klip, transkrip, atau id penutur.
FINDINGS = {}
def rec(section, **kw):
    FINDINGS[section] = {**FINDINGS.get(section, {}), **kw}

def _num(x):
    x = float(x)
    if x == 0.0 or not math.isfinite(x):
        return x
    return float("%.6g" % x)

def jsonable(o):
    if isinstance(o, np.integer):  return int(o)
    if isinstance(o, np.floating): return _num(o)
    if isinstance(o, np.bool_):    return bool(o)
    if isinstance(o, float):       return _num(o)
    if isinstance(o, dict):        return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple, set)): return [jsonable(v) for v in o]
    if isinstance(o, Path):        return str(o)
    return o

def save_findings():
    out("dataset_findings.json").write_text(
        json.dumps(jsonable(FINDINGS), ensure_ascii=False, indent=1, sort_keys=True), encoding="utf-8")

def dump_handoff():
    save_findings()
    blob = json.dumps(jsonable(FINDINGS), ensure_ascii=False, separators=(",", ":"), sort_keys=True)
    print("\n===== BEGIN CLAUDE-HANDOFF-JSON =====")
    print(blob)
    print("===== END CLAUDE-HANDOFF-JSON =====")
    print("size: %.1f KB" % (len(blob) / 1024))

def sha16(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for blk in iter(lambda: fh.read(1 << 22), b""): h.update(blk)
    return h.hexdigest()[:16]

head("DISCOVERY")
INPUT = Path("/kaggle/input")
_hits = sorted(INPUT.glob("**/indonesian_dev/metadata.tsv")) or \
        sorted(p for p in INPUT.glob("**/metadata.tsv") if (p.parent / "clips").is_dir())
if not _hits:
    raise FileNotFoundError("dev set tidak ditemukan - attach dataset indonesian-dev")
DEV_META  = _hits[0]
DEV_CLIPS = DEV_META.parent / "clips"
DEV = pd.read_csv(DEV_META, sep="\t", dtype=str, keep_default_na=False)
print("dev metadata :", DEV_META)
print("dev clips    :", len(list(DEV_CLIPS.glob("*.mp3"))), "mp3 | baris metadata:", len(DEV))
assert len(DEV) == 372, "dev resmi seharusnya 372 klip, dapat %d" % len(DEV)
assert set(DEV["audio_filename"]) == set(p.name for p in DEV_CLIPS.glob("*.mp3")), \
    "metadata dan clips/ tidak cocok satu-satu"
print("convo_id     :", DEV["convo_id"].value_counts().sort_index().to_dict())

head("HUGGINGFACE")
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    HF_TOKEN = os.environ.get("HF_TOKEN")
    if not HF_TOKEN:
        print("tidak ada secret HF_TOKEN (%r)" % (e,))
if not HF_TOKEN and "D" in RUN_ARMS:
    # Kontrol D ada di repo private: tanpa token unduhannya gagal di sel 3.
    raise RuntimeError("HF_TOKEN tidak ada. Add-ons -> Secrets -> HF_TOKEN (per notebook), lalu Run All.")
from huggingface_hub import HfApi
API = HfApi(token=HF_TOKEN)
print("hf user : %s" % API.whoami()["name"])

rec("env", notebook=NOTEBOOK_ID, run_id=RUN_ID, arms=list(RUN_ARMS), num_beams=NUM_BEAMS,
    transformers=transformers.__version__, torch=torch.__version__, numpy=np.__version__,
    python=platform.python_version(), gpu=torch.cuda.get_device_name(0), vram_gb=round(GPU_TOTAL_GB, 1),
    n_dev=len(DEV))
print("\noutput :", OUT_DIR)

## 3 — Checkpoint ketiga arm → `/kaggle/temp`

Hanya file yang dibaca `main.py`. large-v3 juga menyimpan salinan fp32 dan bobot flax/pytorch di repo-nya; semua itu dilewati.

In [ ]:
# ============================================================
# CELL 3 - CHECKPOINT: D (sha dicocokkan), T0, L0
# ============================================================
head("CHECKPOINT")
from huggingface_hub import snapshot_download
from safetensors import safe_open

MAIN_NEEDS = ["config.json", "generation_config.json", "model.safetensors", "tokenizer.json",
              "tokenizer_config.json"]
ALLOW = MAIN_NEEDS + ["preprocessor_config.json", "processor_config.json", "special_tokens_map.json",
                      "added_tokens.json", "normalizer.json", "vocab.json", "merges.txt"]

def _j(p):
    return json.loads(Path(p).read_text(encoding="utf-8")) if Path(p).exists() else {}

def fe_config(d):
    # Sama dengan main.py: preprocessor_config.json (transformers 4) atau processor_config.json
    # dengan feature extractor bersarang (transformers 5, checkpoint D).
    if (d / "preprocessor_config.json").exists():
        return _j(d / "preprocessor_config.json")
    p = _j(d / "processor_config.json")
    return p if "feature_size" in p else next((v for v in p.values() if isinstance(v, dict) and "feature_size" in v), {})

MODEL_DIRS, CKPT = {}, {}
for arm in RUN_ARMS:
    s = ARMS_SPEC[arm]
    dest = TEMP / "ckpt" / arm / "model"
    if dest.parent.exists():
        shutil.rmtree(dest.parent)
    dest.mkdir(parents=True)
    rev = API.model_info(s["repo"], revision=s["rev"]).sha
    t0 = time.time()
    snapshot_download(repo_id=s["repo"], revision=rev, token=HF_TOKEN, local_dir=str(dest), allow_patterns=ALLOW)
    dl = time.time() - t0
    if (dest / ".cache").exists():
        shutil.rmtree(dest / ".cache")
    missing = [n for n in MAIN_NEEDS if not (dest / n).exists()]
    assert not missing, "arm %s: file yang dibaca main.py tidak ada: %s" % (arm, missing)
    got = sha16(dest / "model.safetensors")
    if s["sha16"]:
        assert got == s["sha16"], "arm %s: bobot bukan yang diukur (%s != %s)" % (arm, got, s["sha16"])
    cfg, gen, fe = _j(dest / "config.json"), _j(dest / "generation_config.json"), fe_config(dest)
    has_jw = "<|jw|>" in (gen.get("lang_to_id") or {})
    n_enc = n_dec = 0
    with safe_open(str(dest / "model.safetensors"), "pt") as f:
        dtypes = collections.Counter()
        for k in f.keys():
            sl = f.get_slice(k)
            n = int(np.prod(sl.get_shape()))
            dtypes[str(sl.get_dtype())] += n
            if ".encoder." in k or k.startswith("encoder."): n_enc += n
            elif ".decoder." in k or k.startswith("decoder."): n_dec += n
    CKPT[arm] = dict(repo=s["repo"], revision=rev, sha16=got, sha_ok=(got == s["sha16"]) if s["sha16"] else None,
                     download_s=round(dl, 1), size_gb=round((dest / "model.safetensors").stat().st_size / 1e9, 3),
                     encoder_layers=cfg.get("encoder_layers"), decoder_layers=cfg.get("decoder_layers"),
                     d_model=cfg.get("d_model"), num_mel_bins=cfg.get("num_mel_bins"),
                     feature_size=fe.get("feature_size"), has_jw=has_jw,
                     params_encoder_M=round(n_enc / 1e6, 1), params_decoder_M=round(n_dec / 1e6, 1),
                     weight_dtype=dtypes.most_common(1)[0][0],
                     files=sorted(p.name for p in dest.iterdir() if p.is_file()))
    MODEL_DIRS[arm] = dest
    print("%-3s %s @ %s | %.2f GB %s | enc %s / dec %s layer | %.0f M + %.0f M param | 128 mel %s | <|jw|> %s | unduh %.0f s"
          % (arm, s["repo"], rev[:10], CKPT[arm]["size_gb"], CKPT[arm]["weight_dtype"], cfg.get("encoder_layers"),
             cfg.get("decoder_layers"), n_enc / 1e6, n_dec / 1e6, fe.get("feature_size") == 128, has_jw, dl))
    assert fe.get("feature_size") == cfg.get("num_mel_bins") == 128, "arm %s: bukan model 128 mel" % arm
    assert has_jw, "arm %s: generation_config.json tidak punya <|jw|>" % arm
rec("checkpoints", **CKPT)

## 4 — `main.py` milik D

Salinan byte demi byte dari NB 08 (lewat NB 10), dibuktikan lewat sha. Ketiga arm di-decode oleh file yang sama; hanya folder `model/` yang berbeda.

In [ ]:
# ============================================================
# CELL 4 - main.py DAN postprocess.py, SALINAN BYTE DEMI BYTE DARI NB 08
# ============================================================
head("MAIN.PY")
SRC_MAIN = r'''# main.py - Lost in Transcription, Indonesian-Javanese track.
#
# Runs inside the organiser's container. Working directory is /code_execution:
#   data/clips/*.mp3        read-only test audio
#   data/test_metadata.csv  audio_filename, file_duration_seconds, language
#   src/                    this file, postprocess.py and ./model
#   submission/             where submission.csv must be written
#
# Constraints this file is written against, all taken from the competition's
# own pages rather than assumed:
#   - no network of any kind, so weights ship in ./model and offline flags are
#     set BEFORE transformers is imported (it reads them at import time)
#   - 2 h for the full set, 1 min for the 5-clip smoke test: the smoke budget is
#     import + weight load, not decode, so nothing heavy happens at module scope
#   - logs are capped at 500 lines x 300 chars AND logging anything about the
#     test data is grounds for disqualification, so only aggregate counts and
#     timings are printed - never a filename, a duration, or a transcript
#   - each clip is processed independently (competition rule 4): no state, no
#     sorting by content, no cross-clip normalisation
#
# The decode path is notebook 06's evaluation code (its cells 1 and 7), constant
# for constant: the dev WER this model was chosen on only carries over to the
# test set if the submission decodes exactly the same way.
import os

# Must precede the transformers/HF imports below.
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
os.environ.setdefault("HF_DATASETS_OFFLINE", "1")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
# Library warnings go to stderr, which counts against the same 500-line cap.
os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")

import collections
import csv
import json
import math
import sys
import time
import traceback
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

_T_IMPORT_START = time.time()
import numpy as np
import torch
from transformers import (GenerationConfig, WhisperFeatureExtractor,
                          WhisperForConditionalGeneration, WhisperTokenizerFast)

try:
    import librosa
except Exception:
    librosa = None
_T_IMPORT = time.time() - _T_IMPORT_START

HERE = Path(__file__).resolve().parent
WORK = Path.cwd()
DATA = WORK / "data"
CLIPS = DATA / "clips"
OUT_DIR = WORK / "submission"
MODEL_DIR = HERE / "model"

# Notebook 06, cell 1 (knobs) and cell 7 (decode helpers).
SR = 16000
LANG = "jw"                 # Whisper's Javanese code is "jw", not ISO "jv"
GEN_MAX_LENGTH = 225
NO_REPEAT_NGRAM = 4
BATCH = 8                   # EVAL_BATCH
LONG_S = 29.8               # clips longer than this take the chunked path
CHUNK_S = 28.0              # LONG_CHUNK_S
OVERLAP_S = 2.0             # LONG_OVERLAP_S
MIN_WINDOW_S = 0.3          # a trailing window this short is dropped
MAX_OVERLAP_WORDS = 8       # seam de-duplication, longest match first

# Beam width. 1 = greedy, the setting notebook 06 measured and every constant
# above was chosen under. The packaging notebook rewrites this one line per
# experiment arm and refuses an arm whose main.py differs anywhere else.
NUM_BEAMS = 1

GEN_KWARGS = dict(task="transcribe", language=LANG, num_beams=NUM_BEAMS, do_sample=False,
                  no_repeat_ngram_size=NO_REPEAT_NGRAM, max_length=GEN_MAX_LENGTH,
                  return_timestamps=False)

# A few unreadable clips cost a few points. A run where many clips fail means
# something systemic broke - a library mismatch, a missing file - and would
# hand the platform a scored file of blanks that burns a submission. Past this
# rate no submission.csv is written and the run exits non-zero; failed runs do
# not count against the submission limit.
MAX_FAIL_RATE = 0.02
MAX_FAIL_FLOOR = 2
FAIL_FAST_AFTER = 16

# Set only by our own test notebook. Never set on the platform, where a
# traceback could say something about a test file.
DEBUG = os.environ.get("LIT_DEBUG") == "1"


def log(msg):
    # Aggregate facts only. Never clip identities or transcript content.
    print(msg, flush=True)


def load_audio(path):
    # Notebook 06's load_official_audio: librosa first, soundfile + polyphase
    # resampling as the fallback. The same loader and the same resampler mean
    # the model sees the samples it was measured on.
    if librosa is not None:
        try:
            y, _ = librosa.load(str(path), sr=SR, mono=True)
            return y.astype(np.float32)
        except Exception:
            pass
    import soundfile as sf
    y, sr0 = sf.read(str(path), dtype="float32")
    if y.ndim > 1:
        y = y.mean(axis=1)
    if sr0 != SR:
        from scipy.signal import resample_poly
        g = math.gcd(int(sr0), SR)
        y = resample_poly(y, SR // g, int(sr0) // g)
    return y.astype(np.float32)


def split_windows(y):
    # Fixed windows with an overlap, so a word straddling a boundary is seen
    # whole by at least one window. Whisper's encoder window is 30 s and
    # anything past it would be silently cut.
    step = int((CHUNK_S - OVERLAP_S) * SR)
    win = int(CHUNK_S * SR)
    parts = [y[i:i + win] for i in range(0, len(y), step)]
    return [p for p in parts if len(p) > int(MIN_WINDOW_S * SR)]


def join_overlapping(texts):
    # Consecutive windows usually repeat a few words at the seam. Drop the
    # longest repeated run, longest match first.
    joined = ""
    for t in texts:
        t = (t or "").strip()
        if not t:
            continue
        if not joined:
            joined = t
            continue
        a, b = joined.split(), t.split()
        k = 0
        for n in range(min(MAX_OVERLAP_WORDS, len(a), len(b)), 0, -1):
            if [w.lower() for w in a[-n:]] == [w.lower() for w in b[:n]]:
                k = n
                break
        joined = " ".join(a + b[k:])
    return joined.strip()


def _read_json(path):
    with open(path, encoding="utf-8") as fh:
        return json.load(fh)


FE_KEYS = ("feature_size", "sampling_rate", "hop_length", "chunk_length", "n_fft",
           "padding_value", "dither")


def feature_extractor_config():
    # The checkpoint was saved by transformers 5, which writes a single
    # processor_config.json with the feature extractor nested inside it.
    # transformers 4 - the version the runtime pins - expects a separate
    # preprocessor_config.json. Reading the values by hand makes the file
    # layout of either major version irrelevant.
    pre = MODEL_DIR / "preprocessor_config.json"
    cfg = _read_json(pre) if pre.exists() else _read_json(MODEL_DIR / "processor_config.json")
    if "feature_size" in cfg:
        return cfg
    for v in cfg.values():
        if isinstance(v, dict) and "feature_size" in v:
            return v
    raise RuntimeError("no feature extractor settings found in ./model")


class Recogniser:
    def __init__(self):
        t0 = time.time()
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.dtype = torch.float16 if self.device == "cuda" else torch.float32

        fe_cfg = feature_extractor_config()
        self.fe = WhisperFeatureExtractor(**{k: fe_cfg[k] for k in FE_KEYS if k in fe_cfg})
        # Built straight from tokenizer.json, which both major versions write in
        # the same format; tokenizer_config.json is the file whose format moved.
        # The only thing decoding needs from it is the clean-up flag.
        self.tok = WhisperTokenizerFast(tokenizer_file=str(MODEL_DIR / "tokenizer.json"))
        tok_cfg_path = MODEL_DIR / "tokenizer_config.json"
        tok_cfg = _read_json(tok_cfg_path) if tok_cfg_path.exists() else {}
        self.cleanup = bool(tok_cfg.get("clean_up_tokenization_spaces", False))

        model, info = WhisperForConditionalGeneration.from_pretrained(
            str(MODEL_DIR), local_files_only=True, dtype=self.dtype, output_loading_info=True)
        # With library warnings silenced, a checkpoint whose tensor names do not
        # match would load as random weights without a word. Refuse instead. Two
        # tensors may legitimately be absent from a file saved by another major
        # version: the output projection (tied to the token embedding) and the
        # encoder's fixed sinusoidal positions, which the library recomputes.
        derived = ("proj_out.weight", "encoder.embed_positions.weight")
        missing = info.get("missing_keys", [])
        self.n_derived = sum(1 for k in missing if k.endswith(derived))
        bad = len(missing) - self.n_derived + len(info.get("mismatched_keys", []))
        if bad:
            raise RuntimeError("checkpoint does not fit the model: %d tensors missing or mismatched" % bad)
        self.n_unexpected = len(info.get("unexpected_keys", []))
        # The explicit cast makes the dtype independent of which keyword this
        # transformers release honours (dtype vs torch_dtype).
        self.model = model.to(device=self.device, dtype=self.dtype).eval()
        self.model.config.use_cache = True
        if getattr(self.model.config, "forced_decoder_ids", None) is not None:
            self.model.config.forced_decoder_ids = None
        if self.fe.feature_size != self.model.config.num_mel_bins:
            raise RuntimeError("feature extractor has %d mel bins, model expects %d"
                               % (self.fe.feature_size, self.model.config.num_mel_bins))

        # generation_config.json carries Whisper's language table; without it
        # language="jw" cannot be honoured. Loaded explicitly so that a file this
        # release cannot parse stops the run instead of silently decoding with a
        # default config.
        gc = GenerationConfig.from_pretrained(str(MODEL_DIR), local_files_only=True)
        if "<|%s|>" % LANG not in (getattr(gc, "lang_to_id", None) or {}):
            raise RuntimeError("generation_config.json has no <|%s|> entry" % LANG)
        # Notebook 06's apply_gen_config: language and task pinned, the legacy
        # forced_decoder_ids path switched off so it cannot silently win.
        gc.language = LANG
        gc.task = "transcribe"
        gc.forced_decoder_ids = None
        gc.num_beams = NUM_BEAMS
        gc.do_sample = False
        gc.no_repeat_ngram_size = NO_REPEAT_NGRAM
        gc.max_length = GEN_MAX_LENGTH
        gc.use_cache = True
        self.model.generation_config = gc
        self.t_load = time.time() - t0

    @torch.no_grad()
    def transcribe(self, arrays):
        feats = self.fe(arrays, sampling_rate=SR, return_tensors="pt").input_features
        ids = self.model.generate(feats.to(self.device, dtype=self.dtype), **GEN_KWARGS)
        return [t.strip() for t in self.tok.batch_decode(
            ids, skip_special_tokens=True, clean_up_tokenization_spaces=self.cleanup)]


def read_names():
    meta_path = DATA / "test_metadata.csv"
    if meta_path.exists():
        with open(meta_path, newline="", encoding="utf-8") as fh:
            return [r["audio_filename"] for r in csv.DictReader(fh)]
    # The manifest is documented, but walking clips/ costs nothing and means
    # a missing or renamed manifest degrades instead of failing the run.
    return sorted(p.name for p in CLIPS.glob("*.mp3"))


def main():
    t_start = time.time()
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    names = read_names()
    log("clips to transcribe: %d" % len(names))

    rec = Recogniser()
    log("import %.1fs | model load %.1fs | device %s | %s | tensors recomputed %d, unexpected %d"
        % (_T_IMPORT, rec.t_load, rec.device, str(rec.dtype).replace("torch.", ""),
           rec.n_derived, rec.n_unexpected))

    from postprocess import postprocess

    results = {}
    stats = collections.Counter()
    errors = collections.Counter()
    buf_names, buf_audio = [], []

    def failed(exc):
        stats["failed"] += 1
        errors[type(exc).__name__] += 1
        if DEBUG and errors[type(exc).__name__] == 1:
            traceback.print_exc()

    def run_short():
        # One generate() call per BATCH short clips in manifest order - the batch
        # composition notebook 06 measured. If the batch throws, each clip is
        # retried alone so one bad input cannot blank its neighbours.
        try:
            texts = rec.transcribe(buf_audio)
        except Exception:
            texts = []
            for a in buf_audio:
                try:
                    texts.append(rec.transcribe([a])[0])
                except Exception as e:
                    failed(e)
                    texts.append(None)
        for nm, tx in zip(buf_names, texts):
            results[nm] = "" if tx is None else postprocess(tx)
        stats["short"] += len(buf_names)
        buf_names.clear()
        buf_audio.clear()

    for name in names:
        try:
            y = load_audio(CLIPS / name)
        except Exception as e:
            failed(e)
            results[name] = ""
        else:
            if len(y) / SR > LONG_S:
                # Decoded on the spot, without flushing the short-clip buffer
                # first: notebook 06 did the same, so batches line up clip for clip.
                try:
                    parts = split_windows(y)
                    results[name] = (postprocess(join_overlapping(rec.transcribe(parts)))
                                     if parts else "")
                    stats["chunked"] += 1
                    stats["windows"] += len(parts)
                except Exception as e:
                    failed(e)
                    results[name] = ""
            else:
                buf_names.append(name)
                buf_audio.append(y)
                if len(buf_names) == BATCH:
                    run_short()
        if len(results) >= FAIL_FAST_AFTER and stats["failed"] == len(results):
            raise RuntimeError("the first %d clips all failed (%s) - stopping before a blank file"
                               " is scored" % (len(results), ", ".join(sorted(errors))))
    if buf_names:
        run_short()

    n = len(names)
    n_empty = sum(1 for nm in names if not results.get(nm, "").strip())
    log("short %d | chunked %d (%d windows) | failed %d | empty %d | %d clips in %.1fs"
        % (stats["short"], stats["chunked"], stats["windows"], stats["failed"], n_empty,
           n, time.time() - t_start))
    if errors:
        log("failure types: %s" % ", ".join("%s x%d" % kv for kv in sorted(errors.items())))
    if stats["failed"] > max(MAX_FAIL_FLOOR, int(MAX_FAIL_RATE * n)):
        log("too many failed clips - no submission.csv written, exiting non-zero")
        return 1

    # csv.writer, not string formatting: transcripts contain commas and quotes,
    # and the platform rejects a hand-assembled file.
    out = OUT_DIR / "submission.csv"
    with open(out, "w", newline="", encoding="utf-8") as fh:
        w = csv.writer(fh)
        w.writerow(["audio_filename", "transcript"])
        for nm in names:
            w.writerow([nm, results.get(nm, "")])
    log("wrote %d rows" % n)
    return 0


if __name__ == "__main__":
    sys.exit(main())
'''
SRC_POST = r'''# postprocess.py - text fixes applied to every hypothesis before it is written.
#
# DELIBERATELY ALMOST EMPTY IN PHASE 1. The orthographic rules belong to
# notebook 07, and each one has to earn its place with a paired bootstrap on the
# official dev set before it goes in here. A rule that "looks obviously right"
# is how a submission quietly gets worse: the E4 error table shows the reference
# disagreeing with itself in both directions (gak->nggak 62 times, nggak->gak 33
# times), so a blanket rewrite fixes one column and breaks the other.
#
# What IS safe now is whitespace: it cannot change a token, only how tokens are
# separated, and the official scorer collapses runs of spaces anyway.
#
# Notebook 07 fills RULES below. Keep every entry measurable in isolation.
RULES = []


def postprocess(text):
    if not text:
        return ""
    text = text.replace("\r", " ").replace("\n", " ").replace("\t", " ")
    for pattern, replacement in RULES:
        text = pattern.sub(replacement, text)
    return " ".join(text.split()).strip()
'''
import ast
ast.parse(SRC_MAIN); ast.parse(SRC_POST)

def _sha_text(t):
    return hashlib.sha256(t.encode()).hexdigest()[:16]

MAIN_EVAL, _n_sub = re.subn(r"^NUM_BEAMS = 1$", "NUM_BEAMS = %d" % NUM_BEAMS, SRC_MAIN, flags=re.M)
assert _n_sub == 1, "baris NUM_BEAMS di main.py tidak ditemukan tepat satu kali"
MAIN_BASE_SHA, MAIN_EVAL_SHA = _sha_text(SRC_MAIN), _sha_text(MAIN_EVAL)
print("main.py dasar  sha16 %s  (NB 08: %s) -> %s"
      % (MAIN_BASE_SHA, BASE_MAIN_SHA16, "SAMA" if MAIN_BASE_SHA == BASE_MAIN_SHA16 else "!!! BEDA"))
print("main.py beam %d sha16 %s  (D dikirim: %s) -> %s"
      % (NUM_BEAMS, MAIN_EVAL_SHA, D_MAIN_SHA16,
         "SAMA" if MAIN_EVAL_SHA == D_MAIN_SHA16 else "beda (NUM_BEAMS != 5?)"))
assert MAIN_BASE_SHA == BASE_MAIN_SHA16, "salinan main.py tidak identik dengan NB 08"
if NUM_BEAMS == 5:
    assert MAIN_EVAL_SHA == D_MAIN_SHA16, "main.py evaluasi tidak identik dengan main.py D"
rec("source", main_base_sha16=MAIN_BASE_SHA, main_eval_sha16=MAIN_EVAL_SHA,
    same_as_shipped_D=(MAIN_EVAL_SHA == D_MAIN_SHA16), num_beams=NUM_BEAMS)

## 5 — Scorer resmi, bootstrap berpasangan, dan hitungan S/D/I

`normalize_text` dari `score.py` penyelenggara apa adanya (salinan NB 10 sel 5), ditambah pemecah error menjadi substitusi, penghapusan, dan sisipan.

In [ ]:
# ============================================================
# CELL 5 - SCORER RESMI + BOOTSTRAP BERPASANGAN (salinan NB 10 sel 5) + S/D/I
# ============================================================
def _lowercase_sentence_initial(m):
    delimiter, first, second = m.group(1), m.group(2), m.group(3)
    if first.isupper() and not (second and second.isupper()): first = first.lower()
    return delimiter + first + second

def normalize_text(text):
    BRACKETED = re.compile(r"\[[^\]]+\]"); UNINTELLIGIBLE_PAREN = re.compile(r"\(\?+\)")
    WORD_PAREN = re.compile(r"\(([^()]*)\)"); PUNCTUATION_OTHER = re.compile('[¿¡";:]+')
    COMMA = re.compile(",+"); SENTENCE_INITIAL = re.compile(r"(^\s*|[.!?—]\s*)([^\W\d_])([^\W\d_]?)")
    SENTENCE_END = re.compile("[!?]+"); MULTISPACE = re.compile("  +")
    text = text.replace("~", "")
    text = re.sub(BRACKETED, " ", text); text = re.sub(UNINTELLIGIBLE_PAREN, " ", text)
    text = re.sub(WORD_PAREN, r"\1", text); text = text.replace("#x27;", "'")
    text = re.sub(PUNCTUATION_OTHER, " ", text)
    text = re.sub(SENTENCE_INITIAL, _lowercase_sentence_initial, text)
    text = text.replace("—", ", "); text = re.sub(COMMA, " ", text)
    text = re.sub(SENTENCE_END, " ", text)
    text = text.replace("...", "!ELLIPSIS!").replace(".", " ").replace("!ELLIPSIS!", "...")
    while " ... " in text: text = text.replace(" ... ", " ")
    return re.sub(MULTISPACE, " ", text)

def _lev(r, h):
    prev = list(range(len(h) + 1))
    for i, a in enumerate(r, 1):
        cur = [i] + [0] * len(h)
        for j, b in enumerate(h, 1):
            cur[j] = prev[j - 1] if a == b else 1 + min(prev[j - 1], prev[j], cur[j - 1])
        prev = cur
    return prev[-1]

def corpus_wer(refs, hyps):
    e = n = 0
    for r, h in zip(refs, hyps):
        rt, ht = normalize_text(r).split(), normalize_text(h).split()
        e += _lev(rt, ht); n += len(rt)
    return e / max(n, 1), e, n

def clip_errors(refs, hyps):
    # Per klip: (error, token referensi). Jumlahnya sama dengan corpus_wer; versi per
    # klip dibutuhkan untuk bootstrap berpasangan di bawah.
    out = []
    for r, h in zip(refs, hyps):
        rt, ht = normalize_text(r).split(), normalize_text(h).split()
        out.append((_lev(rt, ht), len(rt)))
    return np.array(out, dtype=np.int64).reshape(-1, 2)

def paired(ex, ey, n_boot=N_BOOT, seed=20260924):
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(ex), size=(n_boot, len(ex)))
    tok = ex[:, 1][idx].sum(1)
    dd = (ex[:, 0][idx].sum(1) - ey[:, 0][idx].sum(1)) / np.maximum(tok, 1)
    lo, hi = np.percentile(dd, [2.5, 97.5])
    point = (ex[:, 0].sum() - ey[:, 0].sum()) / max(ex[:, 1].sum(), 1)
    return dict(delta=round(float(point), 4), ci95=[round(float(lo), 4), round(float(hi), 4)],
                p_better=round(float((dd < 0).mean()), 3))

def lower_nopunct(t):
    # Ruang pembanding kedua: normalizer resmi, lalu huruf kecil semua dan tanpa tanda
    # baca. Kalau S unggul di ruang resmi tapi tidak di sini, keunggulannya kosmetik
    # (case / tanda baca), bukan kata yang didengar.
    return " ".join(re.sub(r"[^\w\s'-]", " ", normalize_text(t).lower()).split())

def clip_errors_ln(refs, hyps):
    out_ = []
    for r, h in zip(refs, hyps):
        rt, ht = lower_nopunct(r).split(), lower_nopunct(h).split()
        out_.append((_lev(rt, ht), len(rt)))
    return np.array(out_, dtype=np.int64).reshape(-1, 2)

REF = dict(zip(DEV["audio_filename"], DEV["transcript"]))
print("scorer resmi siap | contoh normalisasi:", repr(normalize_text("Terus... Aku, ya? Ndak!")))

def lev_ops(r, h):
    # Jarak Levenshtein beserta rinciannya (sub, del, ins). Total selalu = _lev(r, h);
    # rincian dipilih dengan urutan tetap (sub, lalu del, lalu ins) saat biayanya seri.
    prev = [(j, 0, 0, j) for j in range(len(h) + 1)]
    for i, a in enumerate(r, 1):
        cur = [(i, 0, i, 0)] + [None] * len(h)
        for j, b in enumerate(h, 1):
            if a == b:
                cur[j] = prev[j - 1]
                continue
            s, d, n = prev[j - 1], prev[j], cur[j - 1]
            cur[j] = min((s[0] + 1, s[1] + 1, s[2], s[3]), (d[0] + 1, d[1], d[2] + 1, d[3]),
                         (n[0] + 1, n[1], n[2], n[3] + 1), key=lambda t: t[0])
        prev = cur
    return prev[-1]

def corpus_ops(refs, hyps):
    tot = np.zeros(4, dtype=np.int64)
    for r, h in zip(refs, hyps):
        tot += np.array(lev_ops(normalize_text(r).split(), normalize_text(h).split()))
    return dict(errors=int(tot[0]), sub=int(tot[1]), dele=int(tot[2]), ins=int(tot[3]))

assert lev_ops("a b c d".split(), "a x c d e".split()) == (2, 1, 0, 1)
assert lev_ops("a b c".split(), []) == (3, 0, 3, 0) and lev_ops([], "a b".split()) == (2, 0, 0, 2)

## 6 — Decode lewat `main.py` (container tiruan)

Subprocess dengan cwd dan env offline seperti entrypoint resmi, 372 klip dalam urutan NB 08. D pertama: waktu decode arm lain diskalakan dari D untuk memproyeksikan waktu test di A100.

In [ ]:
# ============================================================
# CELL 6 - DECODE D, T0, L0 LEWAT main.py
# ============================================================
head("DECODE LEWAT main.py (beam %d)" % NUM_BEAMS)
import soundfile as sf

def duration_s(p):
    # Hanya untuk kolom file_duration_seconds (main.py tidak membacanya) dan irisan pendek/panjang.
    try:
        return sf.info(str(p)).duration
    except Exception:
        import librosa
        try:
            return librosa.get_duration(path=str(p))
        except TypeError:
            return librosa.get_duration(filename=str(p))

ORDER = list(DEV["audio_filename"])
LANG_OF = dict(zip(DEV["audio_filename"], DEV["language"]))
DUR = {f: duration_s(DEV_CLIPS / f) for f in ORDER}
print("durasi: %d klip, %.2f jam, %d > 29,8 s" % (len(DUR), sum(DUR.values()) / 3600,
                                                    sum(d > 29.8 for d in DUR.values())))

ENV = {k: v for k, v in os.environ.items() if k != "LIT_DEBUG"}
ENV.update(HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1", HF_DATASETS_OFFLINE="1")
CE_ROOT = TEMP / "code_execution"

def build_container(root, model_dir):
    if root.exists():
        shutil.rmtree(root)
    (root / "data" / "clips").mkdir(parents=True)
    (root / "submission").mkdir(parents=True)
    (root / "src").mkdir(parents=True)
    for f in ORDER:
        os.symlink(DEV_CLIPS / f, root / "data" / "clips" / f)
    pd.DataFrame(dict(audio_filename=ORDER,
                      file_duration_seconds=[int(round(DUR[f])) for f in ORDER],
                      language=[LANG_OF[f] for f in ORDER])).to_csv(root / "data" / "test_metadata.csv", index=False)
    (root / "src" / "main.py").write_text(MAIN_EVAL, encoding="utf-8")
    (root / "src" / "postprocess.py").write_text(SRC_POST, encoding="utf-8")
    (root / "src" / "model").symlink_to(model_dir, target_is_directory=True)

def load_times(stdout):
    m = re.search(r"import ([\d.]+)s \| model load ([\d.]+)s", stdout)
    return (float(m.group(1)), float(m.group(2))) if m else (None, None)

def decode_paths(stdout):
    m = re.search(r"short (\d+) \| chunked (\d+) \((\d+) windows\) \| failed (\d+) \| empty (\d+)", stdout)
    return dict(zip(("short", "chunked", "windows", "failed", "empty"), map(int, m.groups()))) if m else {}

RUNS = {}
for arm in RUN_ARMS:
    print("\n--- arm %s: %s ---" % (arm, ARMS_SPEC[arm]["what"]))
    root = CE_ROOT / arm
    build_container(root, MODEL_DIRS[arm])
    t0 = time.time()
    proc = subprocess.run([sys.executable, "src/main.py"], cwd=str(root), env=ENV,
                          capture_output=True, text=True, timeout=3 * 3600)
    wall = time.time() - t0
    log_all = proc.stdout + proc.stderr
    print(proc.stdout[-1200:])
    if proc.returncode != 0:
        print("STDERR:\n", proc.stderr[-4000:])
        rec("run." + arm, exit_code=proc.returncode, stderr_tail=proc.stderr[-600:])
        dump_handoff()
        raise RuntimeError("main.py gagal di arm %s - kirim STDERR dan blok JSON di atas ke Claude" % arm)
    PRED = pd.read_csv(root / "submission" / "submission.csv", keep_default_na=False)
    assert list(PRED.columns) == ["audio_filename", "transcript"], "header submission.csv salah"
    assert list(PRED["audio_filename"]) == ORDER, "daftar/urutan klip submission.csv salah"
    shutil.copy(root / "submission" / "submission.csv", out("submission_dev_%s.csv" % arm))
    imp_s, load_s = load_times(proc.stdout)
    paths = decode_paths(proc.stdout)
    leak = sum(1 for f in ORDER if f.rsplit(".", 1)[0] in log_all)
    n_empty = int((PRED["transcript"].str.strip() == "").sum())
    print("wall %.0f s (%.2f s/klip) | import %s s, load %s s | jalur %s | kosong %d | nama klip di log %d"
          % (wall, wall / len(ORDER), imp_s, load_s, paths, n_empty, leak))
    RUNS[arm] = dict(hyp=dict(zip(PRED["audio_filename"], PRED["transcript"])), paths=paths, wall=wall)
    rec("run." + arm, exit_code=proc.returncode, wall_s=round(wall, 1), s_per_clip=round(wall / len(ORDER), 4),
        import_s=imp_s, load_s=load_s, paths=paths, paths_match=(paths == dict(REF_PATHS, failed=0, empty=0)),
        n_empty=n_empty, leak_names=leak, n_log_lines=len(log_all.splitlines()))
    shutil.rmtree(root, ignore_errors=True)
    save_findings()

## 7 — Skor, paritas D, L0 − T0 berpasangan, proyeksi waktu

Verdict pendengaran mengikuti aturan di atas. Irisan pendek (≤ 29,8 s) paling mirip test: 2.051 dari 2.118 klip test lewat jalur pendek.

In [ ]:
# ============================================================
# CELL 7 - SKOR, PARITAS D, L0 - T0, PROYEKSI
# ============================================================
head("SKOR")
CONVO = DEV["convo_id"].to_numpy()
DURV = np.array([DUR[f] for f in ORDER])
REFS_ORDER = [REF[f] for f in ORDER]
MASKS = {"all": np.ones(len(ORDER), bool), "short": DURV <= 29.8, "long": DURV > 29.8}
for c in sorted(set(CONVO)):
    MASKS[c] = CONVO == c

ERRS, ERRS_LN, SCORES = {}, {}, {}
for arm in RUNS:
    hyps = [RUNS[arm]["hyp"][f] for f in ORDER]
    ERRS[arm] = clip_errors(REFS_ORDER, hyps)
    ERRS_LN[arm] = clip_errors_ln(REFS_ORDER, hyps)
    ops = corpus_ops(REFS_ORDER, hyps)
    hyp_tok = np.array([len(normalize_text(h).split()) for h in hyps])
    ref_tok = ERRS[arm][:, 1]
    runaway = int(((hyp_tok > 2 * ref_tok + 10)).sum())   # tanda loop pengulangan / halusinasi
    SCORES[arm] = dict(ops=ops, len_ratio=round(float(hyp_tok.sum() / max(ref_tok.sum(), 1)), 4),
                       runaway_clips=runaway)
    print("\narm %s  (%s)" % (arm, ARMS_SPEC[arm]["what"]))
    for k, m in MASKS.items():
        e, n = int(ERRS[arm][m, 0].sum()), int(ERRS[arm][m, 1].sum())
        el = int(ERRS_LN[arm][m, 0].sum())
        SCORES[arm][k] = dict(wer=round(e / max(n, 1), 6), n_err=e, n_tok=n,
                              wer_lower_nopunct=round(el / max(n, 1), 6), n_clips=int(m.sum()))
        print("   %-7s WER %.4f (%4d / %5d)   lower+tanpa tanda baca %.4f   [%d klip]"
              % (k, e / max(n, 1), e, n, el / max(n, 1), int(m.sum())))
    print("   S/D/I %d/%d/%d | rasio panjang hyp/ref %.3f | klip runaway %d"
          % (ops["sub"], ops["dele"], ops["ins"], SCORES[arm]["len_ratio"], runaway))

PARITY = {}
if "D" in RUNS:
    print("\nkontrol D vs NB 08 (normalizer resmi):")
    for k, (e0, n0) in D_REF.items():
        got = SCORES["D"][k]
        d = got["wer"] - e0 / n0
        PARITY[k] = dict(n_err=got["n_err"], ref_err=e0, delta_err=got["n_err"] - e0, delta_wer=round(d, 6),
                         ok=bool(got["n_tok"] == n0 and abs(d) <= PARITY_TOL))
        print("   %-4s %4d error vs %4d (NB 08) -> %+d %s"
              % (k, got["n_err"], e0, got["n_err"] - e0, "OK" if PARITY[k]["ok"] else "!!! BERGESER"))

def show(label, res):
    print("   %-44s %+.4f  CI95 [%+.4f, %+.4f]  p(lebih baik) %.3f"
          % (label, res["delta"], res["ci95"][0], res["ci95"][1], res["p_better"]))

COMPARE = {}
print("\nL0 - T0 berpasangan, bootstrap per klip %d resample. Negatif = large-v3 lebih baik." % N_BOOT)
for k in ("all", "short", "long", "2", "5"):
    m = MASKS[k]
    COMPARE["L0_T0_ln_" + k] = paired(ERRS_LN["L0"][m], ERRS_LN["T0"][m])
    COMPARE["L0_T0_off_" + k] = paired(ERRS["L0"][m], ERRS["T0"][m])
    show("%s: lower+tanpa tanda baca (PENDENGARAN)" % k, COMPARE["L0_T0_ln_" + k])
    show("%s: normalizer resmi" % k, COMPARE["L0_T0_off_" + k])
print("\nkonteks (bukan keputusan): model zero-shot vs D yang sudah di-fine-tune")
for arm in ("T0", "L0"):
    COMPARE[arm + "_D_ln_all"] = paired(ERRS_LN[arm], ERRS_LN["D"])
    show("%s - D, lower+tanpa tanda baca" % arm, COMPARE[arm + "_D_ln_all"])

lo, hi = COMPARE["L0_T0_ln_all"]["ci95"]
VERDICT = "L0 LEBIH BAIK" if hi < 0 else "L0 LEBIH BURUK" if lo > 0 else "SERI"
print("\n>>> VERDICT PENDENGARAN (L0 - T0, lower+tanpa tanda baca, 372 klip): %s" % VERDICT)

head("PROYEKSI WAKTU (dari rasio terhadap D)")
PROJ = {}
for arm in RUNS:
    ratio = RUNS[arm]["wall"] / RUNS["D"]["wall"]
    proj_test = KNOWN["D_test_a100_s"] * ratio
    load_s = FINDINGS["run." + arm]["load_s"]
    PROJ[arm] = dict(wall_ratio_vs_D=round(ratio, 3), test_a100_s_proj=round(proj_test),
                     test_ok=bool(proj_test <= TEST_TARGET_S), load_s_t4=load_s,
                     load_ok=bool(load_s is not None and load_s <= LOAD_TARGET_S))
    print("   %-3s %.2fx D -> test A100 ~%5.0f s (target <= %d) %s | load %s s di T4 %s"
          % (arm, ratio, proj_test, TEST_TARGET_S, "OK" if PROJ[arm]["test_ok"] else "!!! LAMBAT",
             load_s, "OK" if PROJ[arm]["load_ok"] else "!!!"))
print("   Kasar: rasio T4 dipakai untuk A100, dan dev punya lebih banyak klip panjang (12 %) dari test (3 %).")

rec("score", per_arm=SCORES, parity_D=PARITY)
rec("compare", verdict_hearing=VERDICT, **COMPARE)
rec("projection", **PROJ)
save_findings()

## 8 — Probe training di T4 (sintetis)

Resep stage 1: adafactor (`scale_parameter=False, relative_step=False`, seperti `optim="adafactor"` di Trainer), fp16 autocast + GradScaler, gradient checkpointing non-reentrant, SpecAugment (0,05 / 10). Satu pemanasan + satu step optimizer dulu (state adafactor dialokasikan di sini), baru diukur. Hasil decode sudah tersimpan: OOM di sel ini tidak menghapus apa pun.

In [ ]:
# ============================================================
# CELL 8 - PROBE TRAINING: VRAM DAN KECEPATAN
# ============================================================
head("PROBE TRAINING (sintetis, label %d token)" % PROBE_LABEL_LEN)
from transformers import WhisperForConditionalGeneration
from transformers.optimization import Adafactor

def _free():
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

def run_probe(arm, variant, mb):
    res = dict(arm=arm, variant=variant, micro_batch=mb, label_len=PROBE_LABEL_LEN)
    model = opt = feats = labels = None
    _free()
    try:
        model = WhisperForConditionalGeneration.from_pretrained(str(MODEL_DIRS[arm]), torch_dtype=torch.float32)
        model.config.use_cache = False
        model.config.forced_decoder_ids = None
        model.config.apply_spec_augment = True
        model.config.mask_time_prob = 0.05
        model.config.mask_time_length = 10
        if variant == "freeze_encoder":
            for p in model.model.encoder.parameters():
                p.requires_grad_(False)
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        model.to("cuda").train()
        params = [p for p in model.parameters() if p.requires_grad]
        res["trainable_M"] = round(sum(p.numel() for p in params) / 1e6, 1)
        opt = Adafactor(params, lr=LEARNING_RATE, scale_parameter=False, relative_step=False, warmup_init=False)
        scaler = torch.cuda.amp.GradScaler()
        g = torch.Generator().manual_seed(SEED)
        feats = torch.randn(mb, model.config.num_mel_bins, 3000, generator=g).cuda()
        labels = torch.randint(100, 50000, (mb, PROBE_LABEL_LEN), generator=g).cuda()

        def micro():
            with torch.autocast("cuda", dtype=torch.float16):
                loss = model(input_features=feats, labels=labels).loss
            scaler.scale(loss).backward()
            return float(loss.detach())

        def step():
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)

        micro(); step(); torch.cuda.synchronize()
        times, losses = [], []
        for _ in range(PROBE_MICRO):
            t = time.time(); losses.append(micro()); torch.cuda.synchronize(); times.append(time.time() - t)
        t = time.time(); step(); torch.cuda.synchronize(); t_step = time.time() - t
        res.update(ok=True, sec_per_micro=round(statistics.median(times), 4), opt_step_s=round(t_step, 3),
                   peak_alloc_gb=round(torch.cuda.max_memory_allocated() / 1e9, 2),
                   peak_reserved_gb=round(torch.cuda.max_memory_reserved() / 1e9, 2),
                   loss_finite=all(math.isfinite(x) for x in losses))
    except torch.cuda.OutOfMemoryError:
        res.update(ok=False, error="OOM", peak_alloc_gb=round(torch.cuda.max_memory_allocated() / 1e9, 2))
    except Exception as e:
        res.update(ok=False, error=repr(e)[:240])
    finally:
        del model, opt, feats, labels
        _free()
    return res

PROBE = []
if RUN_PROBE:
    fitted_full = set()
    for arm, variant, mb in PROBE_CONFIGS:
        if variant == "full" and arm in fitted_full:
            print("   %-3s %-14s mb %d: dilewati (micro-batch lebih besar sudah muat)" % (arm, variant, mb))
            continue
        r = run_probe(arm, variant, mb)
        PROBE.append(r)
        if r.get("ok"):
            if variant == "full" and r["peak_reserved_gb"] <= VRAM_LIMIT_GB:
                fitted_full.add(arm)
            print("   %-3s %-14s mb %d: %.3f s/micro, step %.2f s, puncak %.2f GB (reserved %.2f) | %s M latih | loss %s"
                  % (arm, variant, mb, r["sec_per_micro"], r["opt_step_s"], r["peak_alloc_gb"], r["peak_reserved_gb"],
                     r["trainable_M"], "OK" if r["loss_finite"] else "!!! NaN/inf"))
        else:
            print("   %-3s %-14s mb %d: GAGAL %s" % (arm, variant, mb, r["error"]))
        rec("probe", runs=PROBE)
        save_findings()

    # Kalibrasi: probe turbo vs angka nyata NB 06 (data nyata, micro-batch 4).
    cal = next((r for r in PROBE if r["arm"] == "T0" and r["variant"] == "full" and r["micro_batch"] == 4 and r.get("ok")), None)
    CAL = KNOWN["turbo_sec_per_micro_b4"] / cal["sec_per_micro"] if cal else None
    print("\nkalibrasi: turbo nyata %.3f s/micro vs probe %s -> faktor %s | puncak nyata %.2f GB vs probe %s GB"
          % (KNOWN["turbo_sec_per_micro_b4"], cal and cal["sec_per_micro"], CAL and round(CAL, 3),
             KNOWN["turbo_peak_gb_b4"], cal and cal["peak_alloc_gb"]))
    PROJ_TRAIN = []
    for r in PROBE:
        if not r.get("ok"):
            continue
        n_micro = STAGE1_STEPS * EFF_BATCH / r["micro_batch"]
        raw_h = (n_micro * r["sec_per_micro"] + STAGE1_STEPS * r["opt_step_s"]) / 3600
        cal_h = raw_h * CAL if CAL else None
        fits = r["peak_reserved_gb"] <= VRAM_LIMIT_GB
        PROJ_TRAIN.append(dict(arm=r["arm"], variant=r["variant"], micro_batch=r["micro_batch"],
                               fits_vram=fits, train_h_raw=round(raw_h, 2), train_h_calibrated=cal_h and round(cal_h, 2)))
        print("   %-3s %-14s mb %d: muat %s | stage 1 (%d step) ~%.2f jam mentah, ~%s jam terkalibrasi (tanpa evaluasi)"
              % (r["arm"], r["variant"], r["micro_batch"], "YA" if fits else "TIDAK", STAGE1_STEPS, raw_h,
                 cal_h and "%.2f" % cal_h))
    print("   Pembanding: turbo stage 1 nyata %.2f jam latih saja, %.2f jam total dengan evaluasi."
          % (KNOWN["turbo_stage1_train_only_h"], KNOWN["turbo_stage1_wall_h"]))
    rec("probe", runs=PROBE, calibration_factor=CAL and round(CAL, 3), projection=PROJ_TRAIN)
else:
    print("RUN_PROBE=False: dilewati")
save_findings()

## 9 — Self-validation dan handoff

In [ ]:
# ============================================================
# CELL 9 - SELF-VALIDATION, DIGEST, HANDOFF
# Salin SELURUH blok antara penanda BEGIN/END ke chat.
# ============================================================
head("SELF-VALIDATION")
CHECKS = []
def check(name, cond, detail=""):
    CHECKS.append(dict(check=name, passed=bool(cond), detail=str(detail)[:180]))
    print("  [%s] %-62s %s" % ("PASS" if cond else "FAIL", name, detail))

check("transformers = versi container (<5)", MAJOR < 5, transformers.__version__)
check("bobot D = submission D (sha16 %s)" % ARMS_SPEC["D"]["sha16"], CKPT["D"]["sha_ok"], CKPT["D"]["sha16"])
check("main.py evaluasi = main.py D yang dikirim", FINDINGS["source"]["same_as_shipped_D"],
      FINDINGS["source"]["main_eval_sha16"])
check("T0 turbo: decoder 4 layer", CKPT["T0"]["decoder_layers"] == 4, CKPT["T0"]["decoder_layers"])
check("L0 large-v3: decoder 32 layer", CKPT["L0"]["decoder_layers"] == 32, CKPT["L0"]["decoder_layers"])
for arm in RUN_ARMS:
    r_ = FINDINGS["run." + arm]
    check("arm %s: exit 0, 0 gagal, jalur decode = NB 08" % arm, r_["exit_code"] == 0 and r_["paths_match"], r_["paths"])
    check("arm %s: tidak ada nama klip di log main.py" % arm, r_["leak_names"] == 0, r_["leak_names"])
for k, p in PARITY.items():
    check("kontrol D = NB 08 (%s)" % k, p["ok"], "%+d error" % p["delta_err"])
if RUN_PROBE:
    check("probe kalibrasi turbo jalan", any(r["arm"] == "T0" and r.get("ok") for r in PROBE))
    check("probe large-v3: minimal satu konfigurasi jalan", any(r["arm"] == "L0" and r.get("ok") for r in PROBE))
    check("probe: loss selalu finite", all(r.get("loss_finite", True) for r in PROBE if r.get("ok")))

n_pass = sum(c["passed"] for c in CHECKS)
print("\n%d/%d PASS" % (n_pass, len(CHECKS)))
rec("self_validation", passed=n_pass, total=len(CHECKS), failed=[c["check"] for c in CHECKS if not c["passed"]])
rec("summary", verdict_hearing=VERDICT,
    L0_T0_ln_all=COMPARE["L0_T0_ln_all"], L0_T0_ln_short=COMPARE["L0_T0_ln_short"],
    wer=dict((a, SCORES[a]["all"]["wer"]) for a in RUNS),
    wer_lower_nopunct=dict((a, SCORES[a]["all"]["wer_lower_nopunct"]) for a in RUNS),
    test_a100_s_proj=dict((a, PROJ[a]["test_a100_s_proj"]) for a in RUNS))
dump_handoff()

## Setelah notebook ini

1. **Salin blok `CLAUDE-HANDOFF-JSON`** ke chat (agregat saja), lalu unduh
   `findings/11-large-v3-probe-zeroshot-feasibility/` ke repo seperti biasa. File
   `submission_dev_*.csv` dianalisis lokal oleh skrip yang hanya mencetak agregat.
2. **Check yang paling penting:** *"kontrol D = NB 08"*. Kalau PASS, lingkungan decode sama dengan
   submission D. Perbandingan L0 − T0 tetap sah walau FAIL, karena keduanya di-decode di sesi ini.
3. **Langkah berikutnya menurut aturan di atas:**
   - **L0 LEBIH BAIK** + ada konfigurasi full yang muat dengan proyeksi ≤ 9 jam → NB 06 disiapkan
     untuk stage 1 large-v3 (resep E4-native, `RUN_NOTE` sendiri, bisa resume), lalu NB 10 c5.
   - **SERI** → dibahas: large-v3 tetap satu-satunya tuas besar, tapi tanpa dukungan data.
   - **L0 LEBIH BURUK** → large-v3 ditutup; sisa jalur: turbo 2 epoch (R3).